# 1. Coletar dados

### 1.1 Carregar a rede

In [ ]:
import random

import networkx as nx
import pandas as pd

caminho = "./data/facebook_combined.txt.gz"

In [ ]:
G = nx.read_edgelist(caminho, nodetype=int)

print("Nós:", G.number_of_nodes())
print("Arestas:", G.number_of_edges())

### 1.2 Escolher os 404 nós aleatórios

In [ ]:
# Definindo semente fixa de 42 para que a mesma amostra
# aleatória seja gerada sempre para nossas análises
rng = random.Random(42)

# selecionar 404 usuários aleatoriamente e extrair todas as arestas incidentes a esses usuários,
# inclusive as conexões com usuários que não foram sorteados
nos_selecionados = rng.sample(list(G.nodes()), 404)
# print("Lista de nós selecionados:", nos_selecionados)

# 2. Construir o grafo

### 2.1 Representar a rede com NetworkX

In [ ]:
# OPÇÃO 1:
#  Se for para ter exatamente 404 nós e manter só as arestas que conectam os sorteados, precisa:
G_amostra = G.subgraph(nos_selecionados).copy()
print("Nós:", G_amostra.number_of_nodes())
print("Arestas:", G_amostra.number_of_edges())

# OPÇÃO 2:
# Se for para extrair todas as arestas incidentes aos nós selecionados, e consequentemente
# adicionar os nós aos quais os nós selecionados se conectam:
# arestas_selecionadas = [
#     (u, v)
#     for u, v in G.edges()
#     if u in nos_selecionados or v in nos_selecionados
# ]

# G_amostra = nx.Graph()
# G_amostra.add_edges_from(arestas_selecionadas)
# print("Nós adicionados devido as conexões:", G_amostra.number_of_nodes())
# print("Todas as arestas conectadas aos 404 nós escolhidos:", G_amostra.number_of_edges())

### 2.2 Visualizar o grafo gerado

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 12))

# pos = associa cada nó a uma posição no desenho
# seed=42 ajuda a manter a mesma disposição dos itens no desenho executar novamente
pos = nx.spring_layout(G_amostra, seed=42)

nx.draw(
    G_amostra,
    pos,
    node_size=20,
    node_color="skyblue",
    edge_color="gray",
    with_labels=False,
)

plt.show()

# 3. Calcular métricas

## 3.1 Mapear comunidades pelo algoritmo de Louvain

In [ ]:
## Louvain = identificar grupos de nós que têm mais conexões entre si do que com o resto da rede
comunidades = nx.community.louvain_communities(G_amostra, seed=42)

# Retorna uma lista, em que cada item é um comunidade
print("Lista de comunidades: ", comunidades)
# todo: armazenar a lista de comunidades numa tabela do pandas
print("Quantidade de comunidades:", len(comunidades))

### 3.2 Calcular as medidas de centralidade

In [ ]:
## Grau: quantas conexões um nó possui, normalizadas pelo tamanho do grafo.
grau = nx.degree_centrality(G_amostra)
print("Graus de todos os nós selecionados: ", grau)

In [ ]:
# Intermediação: quanto um nó participa dos caminhos mínimos entre outros nós.
intermediacao = nx.betweenness_centrality(G_amostra)
print("Intermediação:", intermediacao)

In [ ]:
# Proximidade: quão perto um nó está dos demais, considerando os caminhos mínimos.
proximidade = nx.closeness_centrality(G_amostra)
print("Proximidade:")

In [ ]:
# considera não apenas quantas conexões um nó tem, mas também a importância dos nós aos quais está conectado.
autovetor = nx.eigenvector_centrality(G_amostra, max_iter=1000)

In [ ]:
# Função auxiliar para identificar a qual comunidade cada nó pertence
comunidade_por_no = {
    no: i for i, comunidade in enumerate(comunidades) for no in comunidade
}

In [ ]:
# Adicionar as métricas como atributos dos nós
nx.set_node_attributes(G_amostra, comunidade_por_no, "comunidade")
nx.set_node_attributes(G_amostra, grau, "grau")
nx.set_node_attributes(G_amostra, intermediacao, "intermediacao")
nx.set_node_attributes(G_amostra, proximidade, "proximidade")
nx.set_node_attributes(G_amostra, autovetor, "autovetor")

# Exportar o grafo para o Gephi
nx.write_gexf(G_amostra, "./data/grafo_metricas.gexf")

In [ ]:
# Transforma todas as métricas extraídas em formato de Dataframe e salva em um csv
df_metricas = pd.DataFrame(
    {
        "no": list(G_amostra.nodes()),
        "comunidade": [comunidade_por_no[no] for no in G_amostra.nodes()],
        "grau": [grau[no] for no in G_amostra.nodes()],
        "intermediacao": [intermediacao[no] for no in G_amostra.nodes()],
        "proximidade": [proximidade[no] for no in G_amostra.nodes()],
        "autovetor": [autovetor[no] for no in G_amostra.nodes()],
    }
)

df_metricas.head()


df_metricas.to_csv("./data/metricas_grafo.csv", index=False)

# 4. Visualizar as medidas de forma explícita

In [ ]:
# Visualizar as comunidades de Louvain geradas, cada uma com uma cor diferente
cores_comunidades = [comunidade_por_no[no] for no in G_amostra.nodes()]

plt.figure(figsize=(14, 10))

nx.draw_networkx_edges(G_amostra, pos, alpha=0.15, edge_color="gray")

nx.draw_networkx_nodes(
    G_amostra, pos, node_color=cores_comunidades, cmap=plt.cm.tab20, node_size=25
)

plt.title(
    f"Comunidades identificadas pelo algoritmo de Louvain. Quantidade: {len(comunidades)}"
)
plt.axis("off")
plt.show()

In [ ]:
# Função auxiliar para visualizar as medidas de centralidade


def visualizar_centralidade(G, pos, metricas, nome_metrica, top_k=10):
    maiores = sorted(metricas.items(), key=lambda item: item[1], reverse=True)[:top_k]

    valores = [metricas[no] for no in G.nodes()]
    nos_destacados = [no for no, valor in maiores]

    plt.figure(figsize=(18, 14))

    nx.draw_networkx_edges(G, pos, alpha=0.08, edge_color="gray")

    nos = nx.draw_networkx_nodes(
        G, pos, node_color=valores, cmap=plt.cm.plasma, node_size=25
    )

    # Desenha os nós mais importantes por cima dos demais
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=nos_destacados,
        node_color="red",
        node_size=180,
        edgecolors="black",
    )

    rotulos = {no: f"{no}\n{metricas[no]:.4f}" for no in nos_destacados}

    # Para retirar labels só comentar esse pedaço. As labels podem ficar confusas dependendo da img
    nx.draw_networkx_labels(
        G,
        pos,
        labels=rotulos,
        font_size=10,
        font_weight="bold",
        bbox={"facecolor": "white", "edgecolor": "gray", "alpha": 0.9, "pad": 2},
    )

    plt.colorbar(nos, label=f"Centralidade: {nome_metrica}")
    plt.title(f"{nome_metrica} — Top {top_k} nós")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    return maiores

In [ ]:
top_grau = visualizar_centralidade(G_amostra, pos, grau, "Grau")

In [ ]:
top_intermediacao = visualizar_centralidade(
    G_amostra, pos, intermediacao, "Intermediação"
)

In [ ]:
top_proximidade = visualizar_centralidade(G_amostra, pos, proximidade, "Proximidade")

In [ ]:
top_autovetor = visualizar_centralidade(G_amostra, pos, autovetor, "Autovetor")

### 4.1 Visualizar os grafos no navegador com pyvis

In [ ]:
# Gera um grafo interativo da rede base criada para ser visto no navegador
from pyvis.network import Network

grafo_visual = Network(
    height="750px",
    width="100%",
    bgcolor="white",
    font_color="black",
    notebook=True,
    cdn_resources="in_line",
)

# Adicionar nós e arestas do NetworkX
grafo_visual.from_nx(G_amostra)

# # Mostrar o grafo interativo
grafo_visual.write_html("./data/metricas/rede_gerada.html")

In [ ]:
# Função auxiliar que recebe uma métrica, seleciona os 10 nós
# com maiores valores e gera um HTML interativo

import colorsys
from pathlib import Path

from pyvis.network import Network

# Pasta em que os arquivos HTML serão salvos
pasta_saida = Path("./data/metricas")
pasta_saida.mkdir(parents=True, exist_ok=True)


def visualizar_metrica(G, metricas, nome_metrica, top_k=10):
    # Selecionar os nós com os maiores valores
    maiores = sorted(metricas.items(), key=lambda item: item[1], reverse=True)[:top_k]

    nos_destacados = {no for no, valor in maiores}

    menor = min(metricas.values())
    maior = max(metricas.values())

    grafo_visual = Network(
        height="800px",
        width="100%",
        bgcolor="white",
        font_color="black",
        cdn_resources="in_line",
    )

    # Adicionar os nós
    for no in G.nodes():
        valor = metricas[no]

        if maior > menor:
            proporcao = (valor - menor) / (maior - menor)
        else:
            proporcao = 0.5

        # Cor varia conforme o valor da métrica
        cor = colorsys.hsv_to_rgb(0.7 * (1 - proporcao), 0.85, 0.95)
        cor = f"#{int(cor[0] * 255):02x}{int(cor[1] * 255):02x}{int(cor[2] * 255):02x}"

        # Destacar os nós com maiores valores
        destaque = no in nos_destacados

        rotulo = f"{no}\n{valor:.4f}" if destaque else str(no)

        grafo_visual.add_node(
            no,
            label=rotulo,
            title=f"Nó: {no}<br>{nome_metrica}: {valor:.6f}",
            color="red" if destaque else cor,
            size=30 if destaque else 10,
            font={"size": 18 if destaque else 10},
        )

    # Adicionar as arestas
    grafo_visual.add_edges(G.edges())

    # Salvar o arquivo HTML
    nome_arquivo = (
        nome_metrica.lower().replace(" ", "_").replace("ã", "a").replace("ç", "c")
    )

    caminho = pasta_saida / f"{nome_arquivo}.html"

    grafo_visual.write_html(str(caminho))

    print(f"Arquivo gerado: {caminho}")

    return maiores

## 4.2 Grafos interativos gerados em HTML para visualizar as métricas
Explicação dos destaques feitos nos grafos gerados:
- Os nós em vermelho são os 10 maiores valores da métrica selecionada.
- O rótulo desses nós apresenta o ID e o valor, com quatro casas decimais.
- Os outros nós recebem cores conforme os valores da métrica e mostram o ID.
- Ao passar o mouse sobre um nó, da para consultar seu ID e o valor completo da métrica.


In [ ]:
# gera um html para cada métrica
visualizar_metrica(G_amostra, grau, "Grau")

visualizar_metrica(G_amostra, intermediacao, "Intermediacao")

visualizar_metrica(G_amostra, proximidade, "Proximidade")

visualizar_metrica(G_amostra, autovetor, "Autovetor")